# Arbigen · Cuaderno de ciencia de datos

Este cuaderno presenta la secuencia analítica del proyecto con **datos exportados de una investigación guardada**. No ejecuta búsquedas de Apify ni usa credenciales. El ETL, las variables, la segmentación, el pronóstico, la rentabilidad y el Opportunity Score importan las reglas del backend para que los resultados coincidan con la aplicación.

Descarga el dataset desde la ficha de la investigación y, si ya guardaste una hipótesis financiera, descarga su JSON desde «Evaluar rentabilidad». Coloca ambos archivos en `notebooks/data/` o define `ARBIGEN_DATASET_PATH` y `ARBIGEN_SCENARIO_PATH`. El escenario es opcional; no se sustituyen sus importes por datos ficticios.

## 1. Procedencia y límites de las fuentes

1. En «Mis análisis», abre una investigación, importa si corresponde un CSV de Google Trends y usa «Consultar y preparar» **solo si deseas pagar una nueva extracción**. Si ya existe una muestra guardada, no hace falta repetirla.
2. Pulsa **«Descargar datos para el cuaderno»** y guarda el archivo en `notebooks/data/dataset.json` o indica otra ruta con `ARBIGEN_DATASET_PATH`.
3. La muestra de productos es una observación parcial de publicaciones o catálogo; no mide ventas ni cuota de mercado. Trends es un índice relativo 0–100; `<1` es un dato censurado, no cero. Cada país conserva su moneda local.

El JSON exportado contiene los campos internos normalizados y la procedencia de Trends; no contiene el token de Apify. Si `trend_source` es `google_trends_csv_geo_unverified`, el CSV no permite verificar el país desde su contenido: confirma que exportaste la geografía indicada en `country`. Evita publicar el archivo si sus términos de búsqueda o datos te resultan sensibles.


In [ ]:
import json
import os
import sys
from pathlib import Path

from datetime import date
from decimal import Decimal

cwd = Path.cwd().resolve()
repo = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(repo / "backend"))

from app.modules.etl.pipeline import prepare_dataset
from app.modules.marketplace.domain import MarketplaceProduct
from app.modules.trends.domain import TrendObservation
from app.modules.features.engineering import calculate_features

path = Path(os.environ.get("ARBIGEN_DATASET_PATH", str(repo / "notebooks" / "data" / "dataset.json")))
if not path.is_file():
    raise FileNotFoundError(f"Exporta el dataset desde la ficha y guárdalo en {path}, o configura ARBIGEN_DATASET_PATH")
data = json.loads(path.read_text(encoding="utf-8"))
assert data["country"] in {"MX", "CO", "AR"}
print(f"País: {data['country']} · muestra: {len((data.get('snapshot') or {}).get('items', []))} productos · Trends: {len(data.get('trends', []))} puntos")
print("Procedencia Trends:", data.get("trend_source") or "sin serie", "· importado:", data.get("trend_imported_at") or "—")
if data.get("trend_source") == "google_trends_csv_geo_unverified":
    print("Advertencia: el CSV no declara la geografía; verifica manualmente el país de exportación.")


## 2. ETL reproducible

El ETL normaliza títulos, elimina duplicados por ID y conserva para el análisis de precios solo productos con moneda y precio válidos. Las pistas de material salen del título y **no son atributos verificados**. Si el archivo no contiene muestra, el informe de calidad de productos queda sin procedencia; los puntos de Trends se pueden analizar igualmente.


In [ ]:
snapshot = data.get("snapshot")
raw_items = (snapshot or {}).get("items", [])
products = [MarketplaceProduct(
    id=row["id"], title=row["title"],
    price=Decimal(str(row["price"])) if row.get("price") is not None else None,
    currency=row.get("currency"), permalink=row.get("permalink"), image_url=row.get("image_url"),
) for row in raw_items]
observations = [TrendObservation(
    date=date.fromisoformat(row["date"]),
    value=Decimal(str(row["value"])) if row.get("value") is not None else None,
    less_than_one=row["less_than_one"],
) for row in data.get("trends", [])]
prepared = prepare_dataset(country=data["country"], products=products, trends=observations)
quality = prepared.product_quality
print("Entradas:", quality.source_count, "· depurados con precio:", quality.included_count)
print("Duplicados:", quality.duplicate_count, "· sin precio:", quality.missing_price_count)
print("Precio inválido:", quality.invalid_price_count, "· moneda inválida:", quality.invalid_currency_count)
print("Puntos de Trends:", len(prepared.trends), "· censurados <1:", sum(point.less_than_one for point in prepared.trends))


## 3. Ingeniería de variables

Las estadísticas de precio se calculan sobre productos depurados y en una sola moneda. La desviación estándar es **poblacional**. La cobertura usa el número de entradas de la muestra como denominador. Para Trends, cada ventana incluye tres observaciones consecutivas a frecuencia regular (diaria, semanal o mensual); no se imputan valores `<1`. Si faltan datos, la variable queda como `None`.


In [ ]:
features = calculate_features(
    country=data["country"], products=prepared.products, trends=prepared.trends,
    source_count=len(raw_items) if snapshot is not None else None,
)

def display_table(title, rows):
    print("\n" + title)
    print("-" * len(title))
    for label, value in rows:
        print(f"{label:<32} {value if value is not None else 'No disponible'}")

m, t = features.market, features.trend
display_table("MERCADO", [
    ("Moneda", m.currency), ("Entradas de la muestra", m.source_count),
    ("Productos con precio", m.priced_count), ("Cobertura de precios (%)", m.price_coverage_pct),
    ("Precio mínimo", m.price_min), ("Precio mediano", m.price_median),
    ("Precio medio", m.price_mean), ("Precio máximo", m.price_max),
    ("Desviación estándar de precio", m.price_stddev),
    ("Pistas de material", m.material_hints or "No disponibles"),
])
display_table("TRENDS · ÍNDICE RELATIVO", [
    ("Puntos totales", t.point_count), ("Puntos observados", t.observed_count),
    ("Puntos censurados <1", t.censored_count), ("Cobertura observada (%)", t.observed_coverage_pct),
    ("Índice medio", t.mean_index), ("Volatilidad del índice", t.volatility_index),
    ("Media móvil de 3", t.moving_average_3), ("Crecimiento 3 vs. 3 (%)", t.growth_3v3_pct),
    ("Momentum 3 vs. 3", t.momentum_3v3), ("Aceleración 3 vs. 3", t.acceleration_3v3),
])


### 3.1 Distribución de precios de la muestra

La ficha de resultados muestra hasta cinco rangos de precio de la muestra depurada. El rango completo entre el mínimo y el máximo se divide en partes de igual ancho; el extremo máximo entra en la última barra. Si todos los productos tienen el mismo precio, se usa una sola barra. Esta visualización cuenta publicaciones observadas, **no ventas ni demanda**. Las cantidades deben sumar el número de productos con precio válido.


In [ ]:
from math import floor

prices = [float(product.price) for product in prepared.products if product.price > 0]
if prices:
    lowest, highest = min(prices), max(prices)
    number_of_bins = 1 if lowest == highest else min(5, len(set(prices)))
    width = 0 if number_of_bins == 1 else (highest - lowest) / number_of_bins
    counts = [0] * number_of_bins
    for price in prices:
        position = 0 if width == 0 else min(number_of_bins - 1, floor((price - lowest) / width))
        counts[position] += 1

    for index, count in enumerate(counts):
        lower = lowest + index * width
        upper = highest if index == number_of_bins - 1 else lowest + (index + 1) * width
        label = f"{lower:,.2f}" if width == 0 else f"{lower:,.2f}–{upper:,.2f}"
        print(f"{label:>24} {m.currency} | {'█' * max(1, round(count / max(counts) * 25))} {count}")
    assert sum(counts) == len(prices)
else:
    print("No hay productos con precio para mostrar la distribución.")


## 4. Segmentación de productos

Se ejecuta **K-Means determinista** sobre los productos depurados. La variable principal es `log(1 + precio)`, estandarizada dentro de la muestra; las pistas de material del título se codifican como categorías de peso 0,35. No se usan ventas, competidores, embeddings ni PCA porque esos datos o modelos aún no están disponibles y las pistas de texto no están verificadas.

Antes del ajuste, una regla de Tukey (1,5 × rango intercuartílico) puede excluir como atípica una cola pequeña de precios: como máximo el 10 % y solo si quedan al menos ocho productos. El reporte conserva ambos conteos. Se evalúan 2–4 grupos, cada uno con al menos dos productos. Solo se muestran si hay al menos ocho productos, cuatro precios distintos, una amplitud de precios de al menos 15 % respecto a la mediana y una silueta media ≥ 0,20. La silueta (−1 a 1) mide separación interna, **no** valor comercial ni validez fuera de esta muestra. No se elige un «grupo recomendado» hasta tener rentabilidad y señales de demanda verificadas.


In [ ]:
from app.modules.clustering.model import MODEL_VERSION, cluster_products

clusters = cluster_products(country=data["country"], products=prepared.products)
print("Modelo:", MODEL_VERSION, "· productos:", clusters.product_count, "· estado:", clusters.status)
print("Productos agrupados:", clusters.clustered_count, "· precios atípicos excluidos:", clusters.excluded_outlier_count)
if clusters.status == "ready":
    print("Grupos:", clusters.selected_k, "· silueta:", clusters.silhouette)
    for segment in clusters.segments:
        display_table(f"GRUPO {segment.number} · {segment.label}", [
            ("Productos", segment.count), ("Participación (%)", segment.share_pct),
            ("Precio mínimo", segment.price_min), ("Precio mediano", segment.price_median),
            ("Precio medio", segment.price_mean), ("Precio máximo", segment.price_max),
            ("Pistas de material", segment.material_hints or "No disponibles"),
        ])
        print("Ejemplos representativos:", "; ".join(
            f"{item.title} ({item.price} {clusters.currency})" for item in segment.examples))
else:
    print(clusters.reason)


## 5. Serie temporal y validación retrospectiva

Se analiza exclusivamente el **índice relativo** de Google Trends. No equivale a ventas ni a número absoluto de búsquedas. Los puntos `<1` son censurados: no se sustituyen por cero; el modelo solo usa la secuencia observada posterior al último punto censurado. Se exigen al menos **16 puntos consecutivos** con frecuencia diaria, semanal o mensual.

La validación retrocede en el tiempo con ventanas de entrenamiento crecientes (mínimo ocho observaciones), al menos seis orígenes y un horizonte de tres puntos. Se comparan el último valor, la media de tres, una tendencia lineal y, si existen al menos dos ciclos completos, un pronóstico estacional ingenuo. Se elige el menor **MAE** (error absoluto medio) de validación. Si el menor MAE supera 20 puntos del índice, no se publica un pronóstico. Esto compara métodos simples sin seleccionar Prophet, ARIMA o ETS por popularidad; más adelante podrán añadirse si superan estos baselines.

El rango orientativo de cada horizonte usa el percentil 80 de errores retrospectivos con un mínimo de cinco puntos y se limita a 0–100. **No es un intervalo de confianza calibrado** ni garantiza cobertura futura. La dirección reciente compara promedios de tres observaciones y tampoco mide demanda comercial.


In [ ]:
from app.modules.forecasting.model import MODEL_VERSION as FORECAST_MODEL_VERSION, forecast_trends

forecast = forecast_trends(prepared.trends)
print("Modelo:", FORECAST_MODEL_VERSION, "· estado:", forecast.status)
print("Puntos originales:", forecast.source_point_count, "· utilizados:", forecast.used_point_count,
      "· censurados:", forecast.censored_count)
if forecast.candidate_mae:
    display_table("COMPARACIÓN RETROSPECTIVA · MAE EN PUNTOS", [
        (method, error) for method, error in forecast.candidate_mae.items()
    ])
if forecast.status == "ready":
    print("Frecuencia:", forecast.frequency, "· método elegido:", forecast.selected_model,
          "· dirección:", forecast.direction, "· señal estacional:", forecast.seasonal_signal)
    print("Orígenes de validación:", forecast.backtest_origins,
          "· MAE elegido:", forecast.backtest_mae, "· MAE ingenuo:", forecast.naive_mae)
    display_table("PRÓXIMOS TRES PUNTOS · ÍNDICE 0–100", [
        (str(point.date), f"{point.value} (rango {point.lower}–{point.upper})")
        for point in forecast.predictions
    ])
else:
    print("Sin pronóstico publicado:", forecast.reason)


## 6. Límites de las señales disponibles

- Una mediana o dispersión de precio describe **la muestra capturada**, no todo el marketplace. Un producto de catálogo sin precio se conserva en la muestra y se excluye de precios y clustering.
- El crecimiento de Trends compara dos ventanas de tres observaciones; solo tiene sentido dentro de la misma serie, geografía y frecuencia. Un valor alto no equivale a ventas altas.
- `None` significa datos insuficientes, una ventana irregular o un denominador cero. No debe transformarse en cero ni alimentar automáticamente un score.
- Los precios atípicos excluidos no desaparecen del dataset: solo quedan fuera del ajuste y se informan por separado. Los segmentos pueden cambiar si cambia la muestra.
- Un pronóstico solo resume patrones anteriores de una serie relativa; la selección por backtest y el rango orientativo no garantizan el comportamiento futuro. Si se importa otra serie, se recalcula desde cero.


## 7. Rentabilidad por unidad de un escenario guardado

El precio observado es una referencia del marketplace; **no es el costo de adquisición**. Los cinco importes del escenario los registra una persona. Comisión = precio de venta × tasa; costo total = proveedor + envío + otros costos + comisión; utilidad = precio de venta − costo total; margen = utilidad ÷ precio; ROI = utilidad ÷ costo. El precio de equilibrio es el menor centavo que cubre los costos con la comisión indicada. Un denominador cero se representa como `None`. Impuestos, devoluciones y publicidad se incluyen solo si la persona los suma a «Otros gastos».

Si no hay escenario exportado, esta sección y el score permanecen sin supuestos financieros. El JSON puede contener datos comerciales sensibles; `notebooks/data/` está ignorado por Git.

In [ ]:
from app.modules.profitability.model import FinancialAssumptions, calculate_profitability

scenario_path = os.environ.get("ARBIGEN_SCENARIO_PATH")
scenario = None
scenario_export = None
if scenario_path:
    scenario_export = json.loads(Path(scenario_path).read_text(encoding="utf-8"))
    dataset_id = path.stem.removeprefix("arbigen-dataset-")
    if dataset_id != path.stem and scenario_export["analysis_id"] != dataset_id:
        raise ValueError("El dataset y el escenario pertenecen a investigaciones distintas")
    scenario = FinancialAssumptions(**{key: Decimal(str(scenario_export[key])) for key in (
        "sale_price", "product_cost", "shipping_cost", "commission_pct", "other_costs")})
    economics = calculate_profitability(scenario)
    assert economics.unit_profit == Decimal(str(scenario_export["unit_profit"]))
    print("Producto de referencia:", scenario_export["product_title"])
    print("Precio observado:", scenario_export["reference_price"], scenario_export["currency"])
    display_table("RENTABILIDAD POR UNIDAD · escenario guardado", [
        ("Precio de venta", scenario.sale_price), ("Costo del proveedor", scenario.product_cost),
        ("Envío", scenario.shipping_cost), ("Comisión (%)", scenario.commission_pct),
        ("Otros costos", scenario.other_costs), ("Comisión", economics.commission_cost),
        ("Costo total", economics.total_cost), ("Utilidad", economics.unit_profit),
        ("Margen (%)", economics.margin_pct), ("ROI (%)", economics.roi_pct),
        ("Precio de equilibrio", economics.break_even_price),
    ])
else:
    print("Sin escenario guardado. Descarga el JSON desde «Evaluar rentabilidad» y define ARBIGEN_SCENARIO_PATH.")

## 8. Opportunity Score exploratorio y trazabilidad

La versión `exploratory-v2` conserva cuatro componentes de 0–100: crecimiento reciente de Trends, desempeño del pronóstico en pruebas retrospectivas (`100 − 5 × MAE`), margen y ROI. Los pesos por defecto son 0,25 / 0,15 / 0,35 / 0,25. **El componente llamado `stability` en el contrato técnico mide el error histórico del pronóstico, no estabilidad del mercado.** Ocho productos con precio, crecimiento calculable, pronóstico validado y supuestos financieros completos son requisitos mínimos; una operación con pérdida recibe 0. Estos umbrales y escalas son convenciones, no probabilidades ni una calibración con resultados comerciales.

Las advertencias no se esconden detrás del número: muestra pequeña (8–19 productos), cobertura de precios menor al 80 %, geografía no verificable en el CSV, productos capturados hace más de 30 días o serie de Trends antigua para su frecuencia (14 días diaria, 42 semanal, 93 mensual). Son señales de calidad, no factores silenciosos del score. El corte `as_of` se fija explícitamente para reproducir la lectura; la API usa la fecha de consulta.

La sensibilidad calcula **dos hipótesis deterministas**: bajar 10 % el precio de venta, o subir 10 % los costos de proveedor, envío y otros gastos. La comisión porcentual se vuelve a calcular sobre el precio correspondiente. Son pruebas del escenario, no predicciones de precios ni de ventas.


In [ ]:
from app.modules.scoring.model import ScoreWeights, calculate_score

weights = ScoreWeights(growth=Decimal("0.25"), stability=Decimal("0.15"),
                       margin=Decimal("0.35"), roi=Decimal("0.25"))
as_of = date.today()  # fija otra fecha para reproducir un corte histórico
score_report = calculate_score(
    features=features, forecast=forecast, assumptions=scenario, weights=weights,
    trend_source=data.get("trend_source"),
    trend_last_date=prepared.trends[-1].date if prepared.trends else None,
    market_fetched_date=date.fromisoformat(snapshot["fetched_at"][:10]) if snapshot and snapshot.get("fetched_at") else None,
    as_of=as_of,
)
display_table("OPPORTUNITY SCORE EXPLORATORIO · " + score_report.model_version, [
    ("Fecha de evaluación", as_of), ("Estado", score_report.status),
    ("Score 0–100", score_report.score),
    ("Faltantes", ", ".join(score_report.missing) or "Ninguno"),
    ("Productos con precio", score_report.evidence.product_count),
    ("Cobertura de precios (%)", score_report.evidence.price_coverage_pct),
    ("Último dato de Trends", score_report.evidence.trend_last_date),
    ("MAE histórico del pronóstico", score_report.evidence.forecast_mae),
    ("MAE de referencia simple", score_report.evidence.forecast_naive_mae),
    ("Orígenes del backtest", score_report.evidence.forecast_backtest_origins),
    ("Utilidad por unidad", score_report.unit_profit),
    ("Margen (%)", score_report.margin_pct), ("ROI (%)", score_report.roi_pct),
])
print("Componentes 0–100:", score_report.components)
print("Advertencias:", ", ".join(score_report.warnings) or "Ninguna")
for case in score_report.sensitivity:
    print(f"{case.case}: precio {case.sale_price} {score_report.currency} · "
          f"utilidad {case.unit_profit} · score {case.score}")
print("Límites:", *score_report.limitations, sep="\n- ")


## 9. Lectura crítica y datos aún pendientes

- Una mediana de precio describe la muestra capturada; no es un precio de compra ni una garantía de venta. Con menos de 20 publicaciones o cobertura baja, el score requiere especial cautela.
- Trends expresa interés relativo, no demanda absoluta. Si el CSV no declara geografía, verifica manualmente el país. Una serie antigua o de otro país no debería orientar una decisión actual.
- El MAE retrospectivo mide error histórico del índice. La selección del modelo usa el mismo backtest, por lo que el MAE puede ser optimista; no mide riesgo comercial completo.
- Margen, ROI y pruebas de estrés dependen de importes aportados por la persona. Impuestos, devoluciones y publicidad solo entran si se agregan a otros costos.
- El score no se puede calibrar ni validar contra ventas todavía: faltan resultados reales, representatividad de la muestra y mediciones independientes de demanda y competencia. Hasta entonces no es un ranking de oportunidades comprobadas.
- Si se actualiza la muestra, deben revisarse el precio de referencia y los supuestos del escenario antes de volver a interpretar la puntuación.
